In [12]:
import sqlite3
import pandas as pd
import os

from pathlib import Path

DB = "../data/D2020.01.13_S02435_I0764_D.pdb"

conn = sqlite3.connect(DB)

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,WellData
1,BlastomereData
2,GENERAL
3,IMAGES
4,sqlite_stat1


In [16]:
foldername = os.path.join("../data", Path(DB).stem)
print(foldername)
try:
    os.mkdir(foldername)
except FileExistsError:
    pass

../data/D2020.01.13_S02435_I0764_D


In [35]:
welldata = pd.read_sql("SELECT * FROM WellData", conn)
blastomere = pd.read_sql("SELECT * FROM BlastomereData", conn)
general = pd.read_sql("SELECT * FROM GENERAL", conn)

welldata.to_csv(os.path.join(foldername, "WellData.csv"), index=False)
blastomere.to_csv(os.path.join(foldername, "BlastomereData.csv"), index=False)
general.to_csv(os.path.join(foldername, "General.csv"), index=False)

In [17]:
cur = conn.cursor()

cur.execute("""
SELECT
    Well,
    Run,
    Focal,
    Time,
    length(Image),
    hex(substr(Image,1,16))
FROM IMAGES
LIMIT 20;
""")

for row in cur.fetchall():
    print(row)

(1, -1, 3950, -43842.53377460648, 17701, 'FFD8FFE000104A464946000101000001')
(1, -1, 4285, -43842.53377460648, 36120, 'FFD8FFE000104A464946000101000001')
(1, -1, 5003, -43842.53377460648, 17083, 'FFD8FFE000104A464946000101000001')
(2, -1, 3975, -43842.53377460648, 18480, 'FFD8FFE000104A464946000101000001')
(2, -1, 4307, -43842.53377460648, 38018, 'FFD8FFE000104A464946000101000001')
(2, -1, 5018, -43842.53377460648, 18349, 'FFD8FFE000104A464946000101000001')
(3, -1, 4001, -43842.53377460648, 18515, 'FFD8FFE000104A464946000101000001')
(3, -1, 4349, -43842.53377460648, 37467, 'FFD8FFE000104A464946000101000001')
(4, -1, 4029, -43842.53377460648, 18542, 'FFD8FFE000104A464946000101000001')
(4, -1, 4376, -43842.53377460648, 37791, 'FFD8FFE000104A464946000101000001')
(4, -1, 5074, -43842.53377460648, 18441, 'FFD8FFE000104A464946000101000001')
(3, -1, 5056, -43842.53377460648, 18550, 'FFD8FFE000104A464946000101000001')
(1, 1, 45, 43843.46631094907, 24233, 'FFD8FFE000104A464946000101000001')
(1,

In [25]:
from datetime import datetime, timedelta

def ole_to_datetime(x):
    return datetime(1899, 12, 30) + timedelta(days=x)

print(ole_to_datetime(43842.53377460648).date())


2020-01-12


In [34]:
cur = conn.cursor()

outdir = os.path.join(foldername, "extracted_images")
os.makedirs(outdir, exist_ok=True)

metadata = []

cur.execute("""
SELECT rowid, Well, Run, Focal, Time, Image
FROM IMAGES
ORDER BY Well, Run, Time, Focal
""")

count = 0

for rowid, well, run, focal, time, blob in cur:

    well_dir = os.path.join(outdir, f"well_{well:02d}")
    os.makedirs(well_dir, exist_ok=True)

    dt = ole_to_datetime(time)
    day = dt.date()
    clock = dt.time()

    frame = os.path.join(well_dir, f"{run}")
    os.makedirs(frame, exist_ok=True)

    sort_prefix = rowid

    filename = f"{sort_prefix}_{day}_{clock}_{time}_focal{focal:+03d}_{rowid}.jpg"

    filepath = os.path.join(frame, filename)

    with open(filepath, "wb") as f:
        f.write(blob)

    metadata.append({
        "rowid": rowid,
        "well": well,
        "run": run,
        "time": time,
        "focal": focal,
        "filename": str(filepath)
    })

    count += 1

metadata = pd.DataFrame(metadata)
metadata.to_csv(os.path.join(outdir, "images.csv"), index=False)

print(f"Extracted {count} JPEG images.")
metadata.head()

Extracted 7740 JPEG images.


,rowid,well,run,time,focal,filename
0,1,1,-1,-43842.533775,3950,../data/D2020.01.13_S02435_I0764_D/extracted_i...
1,2,1,-1,-43842.533775,4285,../data/D2020.01.13_S02435_I0764_D/extracted_i...
2,3,1,-1,-43842.533775,5003,../data/D2020.01.13_S02435_I0764_D/extracted_i...
3,19,1,1,43843.466311,-45,../data/D2020.01.13_S02435_I0764_D/extracted_i...
4,18,1,1,43843.466311,-30,../data/D2020.01.13_S02435_I0764_D/extracted_i...
